In [1]:
!pip install -q groq pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.9 MB/s eta 0:00:00


In [2]:
import sqlite3
import pandas as pd
import os
from groq import Groq

print("All libraries imported successfully")

All libraries imported successfully


In [3]:
import os
os.environ["GROQ_API_KEY"]=""
client=Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL="llama-3.1-8b-instant"
print("Groq cleint initialised successfully")
print("Using model:",MODEL)

Groq cleint initialised successfully
Using model: llama-3.1-8b-instant


In [4]:
conn = sqlite3.connect("college.db")

print("Database connected successfully")

Database connected successfully


In [6]:
df = pd.read_csv("student_performance.csv")

print(df.head())

   student_id          name  age  gender        department  semester  \
0        1001  Aarav Sharma   19    Male  Computer Science         2   
1        1002   Priya Patel   20  Female  Computer Science         2   
2        1003   Rohit Verma   19    Male       Electronics         2   
3        1004   Sneha Reddy   20  Female        Mechanical         2   
4        1005    Arjun Nair   19    Male  Computer Science         2   

   math_score  science_score  english_score  programming_score  \
0          85             78             72                 91   
1          76             82             88                 79   
2          65             74             61                 55   
3          70             80             75                 48   
4          92             88             81                 95   

   attendance_percentage       city  admission_year  
0                     92     Mumbai            2023  
1                     87  Ahmedabad            2023  
2       

In [7]:
df.to_sql(
    "students",
    conn,
    if_exists="replace",
    index=False
)

print("Data loaded successfully")

Data loaded successfully


In [8]:
pd.read_sql_query(
    "SELECT * FROM students LIMIT 5",
    conn
)

,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023


In [9]:
def get_schema(conn, table_name="students"):

    cursor = conn.cursor()

    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()

    schema_lines = [f"Table: {table_name}"]
    schema_lines.append("Columns:")

    for col in columns:
        schema_lines.append(f"{col[1]} ({col[2]})")

    cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
    sample_rows = cursor.fetchall()

    schema_lines.append("\nSample rows:")

    for row in sample_rows:
        schema_lines.append(str(row))

    return "\n".join(schema_lines)

In [10]:
schema = get_schema(conn)

print(schema)

Table: students
Columns:
student_id (INTEGER)
name (TEXT)
age (INTEGER)
gender (TEXT)
department (TEXT)
semester (INTEGER)
math_score (INTEGER)
science_score (INTEGER)
english_score (INTEGER)
programming_score (INTEGER)
attendance_percentage (INTEGER)
city (TEXT)
admission_year (INTEGER)

Sample rows:
(1001, 'Aarav Sharma', 19, 'Male', 'Computer Science', 2, 85, 78, 72, 91, 92, 'Mumbai', 2023)
(1002, 'Priya Patel', 20, 'Female', 'Computer Science', 2, 76, 82, 88, 79, 87, 'Ahmedabad', 2023)
(1003, 'Rohit Verma', 19, 'Male', 'Electronics', 2, 65, 74, 61, 55, 78, 'Delhi', 2023)


In [11]:
def generate_sql(user_question,schema_text,client,model):
  """
  sends the user's question and database schema to the Groq LLM.
  the LLM generates a sql quesry that answers """

  system_prompt=f""" you are an expert SQL assistant.
  you are connected to sqlite database with the following structure[schema_structure]
  rules you must follow:
  1.generate only a valid sqlite sql query
  2.do not include any explanation or text-only the sql query
  3.do not use markdown code blocks.return the raw sql query
  4.the table name is students
  5.only use column names that exis in the schema above
  6.use single quotes for string values in where cluauses
  7.if the user asks for top n,use order by marks desc limit n"""

  response=client.chat.completions.create(model=model,
                       messages=[
                           {"role":"system","content":system_prompt},
                           {"role":"user","content":user_question}
                       ],temperature=0.0)
  sql_query=response.choices[0].message.content.strip()
  return sql_query
question="Show me all the female students"
print("Question",question)
print("Generting SQL....")

Question Show me all the female students
Generting SQL....


In [12]:
question = "Show me all female students"

print("Question:", question)
print("Generating SQL...")

Question: Show me all female students
Generating SQL...


In [13]:
sql = generate_sql(
    question,
    schema,
    client,
    MODEL
)

print("Generated SQL:")
print(sql)

Generated SQL:
SELECT * FROM students WHERE gender = 'F'


In [15]:
def execute_sql(sql_query, conn):

    try:
        result_df = pd.read_sql_query(
            sql_query,
            conn
        )

        return result_df, None

    except Exception as e:

        return None, str(e)

In [16]:
results, error = execute_sql(
    sql,
    conn
)

print(results)

Empty DataFrame
Columns: [student_id, name, age, gender, department, semester, math_score, science_score, english_score, programming_score, attendance_percentage, city, admission_year]
Index: []


In [19]:
def text_to_sql_agent(question, conn, client, model):

    print("=" * 60)
    print("QUESTION:", question)

    # Step 1: Read schema
    schema = get_schema(conn)

    # Step 2: Generate SQL
    sql = generate_sql(
        question,
        schema,
        client,
        model
    )

    print("\nGenerated SQL:")
    print(sql)

    # Step 3: Execute SQL
    results, error = execute_sql(
        sql,
        conn
    )

    if error:
        print("\nError:", error)
        return None, error

    print("\nQuery Results:")
    print(results)

    return results, None

In [21]:
print(get_schema)

<function get_schema at 0x7ea69386e980>


In [22]:
print(generate_sql)

<function generate_sql at 0x7ea69386ee80>


In [23]:
print(execute_sql)

<function execute_sql at 0x7ea69369de40>


In [24]:
print(text_to_sql_agent)

<function text_to_sql_agent at 0x7ea69386ec00>


In [26]:
def text_to_sql_agent(question, conn, client, model):

    schema = get_schema(conn)

    sql = generate_sql(
        question,
        schema,
        client,
        model
    )

    print("Generated SQL:")
    print(sql)

    results, error = execute_sql(
        sql,
        conn
    )

    if error:
        return None, error

    return results, None

In [27]:
print(text_to_sql_agent)

<function text_to_sql_agent at 0x7ea69364fb00>


In [29]:
import re

In [30]:
import pandas as pd
import sqlite3
import os
import re

In [31]:
result, error = text_to_sql_agent(
    "Show all female students",
    conn,
    client,
    MODEL
)

print(result)

Generated SQL:
SELECT * FROM students WHERE gender = 'F'
Empty DataFrame
Columns: [student_id, name, age, gender, department, semester, math_score, science_score, english_score, programming_score, attendance_percentage, city, admission_year]
Index: []


In [32]:
result, error = text_to_sql_agent(
    "Show top 5 students in Programming",
    conn,
    client,
    MODEL
)

print(result)

Generated SQL:
SELECT * FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5
None


In [33]:
result, error = text_to_sql_agent(
    "How many male and female students are there?",
    conn,
    client,
    MODEL
)

print(result)

Generated SQL:
SELECT COUNT(CASE WHEN gender = 'male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'female' THEN 1 END) AS female_count 
FROM students
   male_count  female_count
0           0             0


In [34]:
result, error = text_to_sql_agent(
    "Show female students who scored above 85 in Science",
    conn,
    client,
    MODEL
)

print(result)

Generated SQL:
SELECT * FROM students WHERE gender = 'F' AND science > 85
None


In [35]:
result, error = text_to_sql_agent(
    "Show female students who scored above 85 in Science or Programming, ordered by marks",
    conn,
    client,
    MODEL
)

print(result)

Generated SQL:
SELECT * FROM students WHERE gender = 'female' AND marks > 85 AND subject IN ('Science', 'Programming') ORDER BY marks DESC
None


In [36]:
result, error = text_to_sql_agent(
    "Show top 3 students in Mathematics",
    conn,
    client,
    MODEL
)

print(result)

Generated SQL:
SELECT * FROM students ORDER BY math_marks DESC LIMIT 3
None


In [37]:
result, error = text_to_sql_agent(
    "How many students are there?",
    conn,
    client,
    MODEL
)

print(result)

Generated SQL:
SELECT COUNT(*) FROM students
   COUNT(*)
0        30


In [38]:
def text_to_sql_agent(user_question, conn, client, model):

    # Step 1: Get schema
    schema_text = get_schema(conn)

    # Step 2: Generate SQL
    print("Generating SQL...")
    generated_sql = generate_sql(
        user_question,
        schema_text,
        client,
        model
    )

    print(f"SQL: {generated_sql}")

    # Step 3: Execute SQL
    result_df, error = execute_sql(
        generated_sql,
        conn
    )

    if error:
        print(f"Error executing SQL: {error}")
        return

    # Step 4: Show results
    print(f"\nData ({len(result_df)} rows returned):")
    print(result_df)

    return result_df

In [39]:
result = text_to_sql_agent(
    "Show all female students",
    conn,
    client,
    MODEL
)

Generating SQL...
SQL: SELECT * FROM students WHERE gender = 'F'

Data (0 rows returned):
Empty DataFrame
Columns: [student_id, name, age, gender, department, semester, math_score, science_score, english_score, programming_score, attendance_percentage, city, admission_year]
Index: []


In [40]:
result = text_to_sql_agent(
    "Show top 5 students in Programming",
    conn,
    client,
    MODEL
)

Generating SQL...
SQL: SELECT * FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5
Error executing SQL: Execution failed on sql 'SELECT * FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5': no such column: subject


In [41]:
result = text_to_sql_agent(
    "How many male and female students are there?",
    conn,
    client,
    MODEL
)

Generating SQL...
SQL: SELECT COUNT(CASE WHEN gender = 'male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'female' THEN 1 END) AS female_count 
FROM students

Data (1 rows returned):
   male_count  female_count
0           0             0


In [42]:
result = text_to_sql_agent(
    "Show female students who scored above 85 in Science or Programming, ordered by marks",
    conn,
    client,
    MODEL
)

Generating SQL...
SQL: SELECT * FROM students WHERE gender = 'female' AND marks > 85 AND subject IN ('Science', 'Programming') ORDER BY marks DESC
Error executing SQL: Execution failed on sql 'SELECT * FROM students WHERE gender = 'female' AND marks > 85 AND subject IN ('Science', 'Programming') ORDER BY marks DESC': no such column: marks
